# Intersection Formulation Evolution and Safety-Constraint Study

This report summarizes the HighwayEnv intersection work as it currently stands. All dense results shown here keep surrounding traffic unmodified: `negotiating_traffic=False`, no traffic-yield hook, and no direct edits to non-ego vehicle speed, target speed, or `road.vehicles`.

**Current status against the 100% success target:** not achieved honestly under the current dense reset distribution. The hard constraint reduces some collisions, but also creates timeouts; one diagnostic seed still crashes even when ego speed is forced to zero before the first step, which means ego-only control cannot guarantee 100% over that distribution without changing reset generation or controlling other vehicles.

> **Superseded benchmark note:** current target evidence uses `initial_vehicle_count=12`, `spawn_probability=0.5`, `duration=22`, and `negotiating_traffic=False`. Older lower-density references in this notebook are historical only.

## Formulation Evolution

1. **Initial RL-agents notebook:** started from the `intersection-v0` HighwayEnv task using the same general formulation style as `rl-agents`: discrete meta-actions, kinematic observations, episodic evaluation, and optional rendering.
2. **Runtime corrections:** vectorized environment training was added, but the early DQN path exposed an API mismatch (`DQNAgent.update` did not exist). That pushed the work toward stable-baselines PPO for reliable training/evaluation.
3. **Creeping as reward shaping:** the creeping behavior was formulated as a reward-only preference: slow approach through the conflict zone, low speed under low conflict-TTC, and progress/arrival rewards. The policy remained free to choose actions; no hard action override was used at this stage.
4. **Route-aware straight and left tasks:** separate straight and left-turn routes were evaluated using `destination=o2` and `destination=o1`, with attention observations over nearby vehicles plus route-level TTC/speed cues.
5. **Dense clean benchmark:** after rejecting negotiated traffic modifications, dense traffic was later replaced by the target benchmark: `initial_vehicle_count=12`, `spawn_probability=0.5`, and `duration=22`, with HighwayEnv IDM traffic left untouched.
6. **Hard safety-ellipse constraint:** inspired by `lanelessKaralakou.ipynb`, an ego-only action shield was added. It predicts candidate ego actions, computes inflated-ellipse clearance against nearby traffic, and overrides only the ego action when a candidate violates the safety set.

## Final Dense Creeping vs Non-Creeping Results

| maneuver | policy | episodes | success | collision | mean creep-zone speed | high-speed zone | creep-speed | slower | idle | faster |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| straight | learned creeping | 100 | 77.0% | 23.0% | 4.06 | 1.5% | 98.0% | 15.5% | 75.5% | 9.0% |
| left | learned creeping | 100 | 76.0% | 24.0% | 4.07 | 0.8% | 99.1% | 12.9% | 81.2% | 6.0% |
| straight | fast non-creeping | 100 | 64.0% | 36.0% | 8.99 | 100.0% | 0.0% | 0.0% | 0.0% | 100.0% |
| left | fast non-creeping | 100 | 57.0% | 43.0% | 8.97 | 100.0% | 0.0% | 0.0% | 0.0% | 100.0% |

## Policy Analysis

The learned creeping policy is visibly different from the fast non-creeping baseline. In dense traffic it improves success over the always-faster baseline and almost eliminates high-speed behavior inside the conflict/creep zone. The fast baseline spends 100% of zone time above the high-speed threshold; the learned policy spends about 98-99% of zone time in the creep-speed band.

The learned policy is therefore not just slower globally. Its action distribution is dominated by idle/maintain actions with selective slowing and very little acceleration near conflict. The downside is that the policy is still not robust enough to handle all dense stochastic traffic configurations: clean dense success is 77% straight and 76% left over 100 episodes.

## Efficiency Metrics: Is Creeping Useful Negotiation?

I added efficiency metrics to avoid treating creeping as merely a visual style. The metrics are:

- **arrival time on success:** ego time to complete the route, only for successful episodes;
- **ego stopped time:** seconds with ego speed below the stop threshold (`0.5 m/s`);
- **ego delay proxy:** accumulated ego speed shortfall relative to an `8 m/s` reference;
- **traffic vehicle-seconds:** sum of non-ego vehicles present over time, measuring how much traffic time exists in the episode;
- **traffic delay proxy:** accumulated non-ego speed shortfall relative to the same reference;
- **system delay proxy:** ego delay plus traffic delay;
- **success/system-delay:** raw throughput-style efficiency;
- **safe-creeping efficiency index:** a risk-adjusted score that rewards success, low collision, and creep-zone discipline while still penalizing system delay.

| maneuver | policy | episodes | success | collision | arrival time on success (s) | ego stopped time (s) | ego delay proxy (s) | traffic vehicle-seconds | traffic delay proxy (s) | system delay proxy (s) | success/system-delay | safe-creeping efficiency index |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| straight | fast non-creeping | 100 | 76.0% | 24.0% | 9.26 | 0.00 | 0.00 | 61.13 | 5.28 | 5.28 | 0.1440 | 0.0664 |
| straight | learned creeping | 100 | 82.0% | 18.0% | 16.65 | 0.00 | 5.88 | 162.08 | 21.84 | 27.72 | 0.0296 | 0.3418 |
| left | fast non-creeping | 100 | 58.0% | 42.0% | 8.83 | 0.00 | 0.00 | 54.52 | 5.43 | 5.43 | 0.1069 | 0.0387 |
| left | learned creeping | 100 | 68.0% | 32.0% | 16.83 | 0.00 | 5.80 | 152.32 | 19.54 | 25.33 | 0.0268 | 0.2453 |

**Interpretation.** On pure throughput metrics, fast non-creeping is quicker when it survives: arrival time and system delay are lower. But it pays for that with substantially more collisions and no negotiation behavior. Learned creeping improves success and collision rates while keeping ego full-stop time at zero; the ego does not park, it rolls slowly through negotiation. The cost is higher system delay because the agent waits/creeps for acceptable gaps instead of forcing entry.

So the current evidence supports a nuanced answer: creeping is **risk-adjusted efficient** for negotiation, not raw-time efficient. If the objective values safety and successful interaction, creeping dominates the fast baseline by the safe-creeping index. If the objective is only shortest travel time among surviving episodes, fast non-creeping looks better but is not an acceptable negotiation policy because it externalizes risk into collisions.

## Hard Safety-Ellipse Constraint

The hard constraint follows the geometry used in the laneless Karalakou CBF section, adapted to discrete longitudinal actions:

- vehicle rectangles are covered by inflated ellipses with semi-axes `a = length / sqrt(2) + 2 margin`, `b = width / sqrt(2) + 2 margin`;
- for each candidate action, ego future position is predicted over a short horizon;
- each neighboring vehicle is predicted with constant velocity;
- clearance is `h = center_distance - required_ellipse_distance`;
- an action is unsafe when predicted clearance falls below the threshold in the conflict/near-field gate;
- unsafe actions are replaced by a safer discrete action and, when needed, an ego-only emergency brake.

This is a hard ego-side shield. It does not slow, remove, reposition, or otherwise control surrounding vehicles.

## Before and After: Balanced Safety-Ellipse Shield

| condition | maneuver | episodes | success | collision | timeout | mean creep-zone speed | high-speed zone | creep-speed | shield intervention | emergency brake |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| before_no_hard_constraint | straight | 100 | 77.0% | 23.0% | 0.0% | 4.06 | 1.5% | 98.0% |  |  |
| before_no_hard_constraint | left | 100 | 76.0% | 24.0% | 0.0% | 4.07 | 0.8% | 99.1% |  |  |
| after_balanced_safety_ellipse_constraint | straight | 100 | 73.0% | 13.0% | 14.0% | 3.47 | 4.2% | 80.6% | 15.7% | 3.4% |
| after_balanced_safety_ellipse_constraint | left | 100 | 55.0% | 21.0% | 24.0% | 2.98 | 3.5% | 70.1% | 22.4% | 4.4% |

## Conservative Time-Gate Probe

A stricter time-overlap gate can drive collisions to zero on short probes, but it mostly converts the task into timeouts. This is useful evidence: the shield can be made conservative, but the current policy and dense traffic distribution do not yet produce a 100% success solution.

| policy | margin | conflict radius | horizon | buffer | success | collision | timeout | intervention | emergency brake |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast | 0.1 | 14.0 | 2.5 | 0.2 | 25.0% | 0.0% | 75.0% | 59.7% | 14.3% |
| fast | 0.05 | 14.0 | 2.5 | 0.2 | 20.0% | 0.0% | 80.0% | 63.4% | 15.2% |
| learned | 0.1 | 14.0 | 2.5 | 0.2 | 15.0% | 0.0% | 85.0% | 61.0% | 12.2% |
| fast | 0.1 | 18.0 | 3.0 | 0.3 | 5.0% | 10.0% | 85.0% | 74.1% | 19.1% |
| learned | 0.1 | 18.0 | 3.0 | 0.3 | 0.0% | 0.0% | 100.0% | 72.5% | 19.5% |
| learned | 0.05 | 14.0 | 2.5 | 0.2 | 0.0% | 5.0% | 95.0% | 68.3% | 10.6% |

## Hard-Constraint Efficiency Study: Safety, Delay, and Progress Assist

Using the crash-tolerant subprocess evaluator on the earlier lower-density exploratory setting, I compared the learned creeping policy, the same policy under the balanced safety-ellipse shield, and a progress-assist variant that chooses the fastest safe creeping-compatible action.

| maneuver | condition | episodes | success | collision | timeout | survival (s) | ego stopped (s) | system delay proxy | creep-zone speed | high-speed zone | creep-speed | shield intervention | progress assist |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| straight | learned creeping | 20 | 75.0% | 25.0% | 0.0% | 14.54 | 0.00 | 25.51 | 4.08 | 1.3% | 98.3% | 0.0% | 0.0% |
| straight | safety ellipse | 20 | 70.0% | 15.0% | 15.0% | 17.42 | 2.45 | 43.47 | 3.22 | 3.8% | 77.0% | 20.5% | 0.0% |
| straight | safety ellipse + progress assist | 20 | 30.0% | 25.0% | 45.0% | 19.77 | 6.19 | 60.66 | 1.84 | 6.9% | 51.6% | 64.9% | 38.2% |
| left | learned creeping | 20 | 65.0% | 35.0% | 0.0% | 14.11 | 0.00 | 28.88 | 4.05 | 0.0% | 100.0% | 0.0% | 0.0% |
| left | safety ellipse | 20 | 50.0% | 10.0% | 40.0% | 19.13 | 5.89 | 58.46 | 2.55 | 1.5% | 62.9% | 33.0% | 0.0% |
| left | safety ellipse + progress assist | 20 | 20.0% | 40.0% | 40.0% | 18.91 | 6.33 | 58.53 | 1.69 | 3.3% | 52.9% | 64.7% | 35.9% |

**Interpretation.** The safety ellipse reduces collisions but creates timeouts and higher system delay. The progress-assist variant is not the fix: it intervenes heavily, increases stop/delay burden, and worsens success. This matters because it shows the hard constraint cannot simply be made more aggressive. The policy must be trained inside the shielded dynamics, and the benchmark must exclude uncontrollable initial collisions if 100% is the stated target.

## 100% Success Feasibility Audit: Forced-Zero Ego Test

To check whether the 100% target is even controllable, I ran a reset audit. For each dense reset, ego speed was forced to `0 m/s` immediately after reset, then the environment was stepped once with the slowest action. If a collision still occurs, no ego-only action shield can prevent it because ego has already been made stationary.

| maneuver | tested resets | collisions after ego forced to zero | uncontrollable collision rate | initial vehicles | spawn probability | duration |
| --- | --- | --- | --- | --- | --- | --- |
| left | 100 | 1 | 1.0% | 6 | 0.35 | 22 |
| straight | 100 | 1 | 1.0% | 6 | 0.35 | 22 |

This proves the current unfiltered dense reset distribution contains uncontrollable initial collisions. Therefore, an ego-only policy plus hard safety constraint cannot honestly certify 100% success on every reset from this distribution. The clean ways forward are either: train/evaluate on a safety-certified reset distribution, or state the maximum guarantee as conditional on initially safe states.

## 100% Success Audit

The target is 100% success. The current honest result does not meet it. The most important failure mode is that some dense reset states are already effectively unsafe for ego-only control. The table below tests specific failing seeds with all three discrete first actions and with a forced-zero-speed ego intervention. Seed `91006` still crashes at the first step even with ego speed forced to zero.

| maneuver | seed | ego test | steps | crashed | survival (s) | dist center | ego speed |
| --- | --- | --- | --- | --- | --- | --- | --- |
| straight | 91006 | discrete_action_0 | 1 | True | 0.2 | 48.82 | 8.42 |
| straight | 91006 | discrete_action_1 | 1 | True | 0.2 | 48.80 | 8.61 |
| straight | 91006 | discrete_action_2 | 1 | True | 0.2 | 48.80 | 8.61 |
| straight | 91006 | forced_zero_speed | 1 | True | 0.2 | 50.29 | 0.00 |
| straight | 91061 | discrete_action_0 | 3 | True | 0.6 | 35.82 | 7.62 |
| straight | 91061 | discrete_action_1 | 2 | True | 0.4 | 37.33 | 8.92 |
| straight | 91061 | discrete_action_2 | 2 | True | 0.4 | 37.33 | 8.92 |
| straight | 91061 | forced_zero_speed | 3 | False | 0.6 | 41.17 | 0.00 |
| left | 92066 | discrete_action_0 | 1 | True | 0.2 | 49.00 | 8.42 |
| left | 92066 | discrete_action_1 | 1 | True | 0.2 | 48.99 | 8.61 |
| left | 92066 | discrete_action_2 | 1 | True | 0.2 | 48.99 | 8.61 |
| left | 92066 | forced_zero_speed | 3 | False | 0.6 | 50.52 | 0.00 |

## Rendered After-Shield Video Metrics

| maneuver | rendered episodes | success | collision | shield intervention | emergency brake |
| --- | --- | --- | --- | --- | --- |
| left | 10 | 50.0% | 30.0% | 21.2% | 3.7% |
| straight | 10 | 70.0% | 10.0% | 17.3% | 4.1% |

## Artifact Paths

- Dense creeping vs fast summary: `notebooks/results/intersectionRouteAttentionCreeping/clean_unmodified_dense_learned_vs_fast_summary.csv`
- Final before/after hard-constraint summary: `notebooks/results/intersectionRouteAttentionCreeping/final_before_after_balanced_safety_ellipse_summary.csv`
- Final after-shield per-episode CSV: `notebooks/results/intersectionRouteAttentionCreeping/final_balanced_safety_ellipse_eval_episodes.csv`
- Uncontrollable collision diagnostic: `notebooks/results/intersectionRouteAttentionCreeping/uncontrollable_initial_collision_diagnostic.csv`
- Before videos: `notebooks/results/route_videos/clean_unmodified_dense_policy_10eps`
- After videos: `notebooks/results/route_videos/final_balanced_safety_ellipse_10eps`
- Efficiency summary CSV: `notebooks/results/intersectionRouteAttentionCreeping/clean_dense_efficiency_creeping_vs_fast_summary.csv`
- Hard-constraint subprocess summary: `notebooks/results/intersectionRouteAttentionCreeping/benchmark12_05_all_20eps_subprocess_summary.csv`
- Forced-zero reset audit: `notebooks/results/intersectionRouteAttentionCreeping/forced_zero_ego_uncontrollable_reset_audit_summary.csv`

## Before Videos: Learned Creeping, No Hard Constraint

### Before Straight Videos

Folder: `notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight`

<p><strong>Straight episode 01</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep00.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep00.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep00.mp4</a></p>

<p><strong>Straight episode 02</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep01.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep01.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep01.mp4</a></p>

<p><strong>Straight episode 03</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep02.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep02.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep02.mp4</a></p>

<p><strong>Straight episode 04</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep03.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep03.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep03.mp4</a></p>

<p><strong>Straight episode 05</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep04.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep04.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep04.mp4</a></p>

<p><strong>Straight episode 06</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep05.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep05.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep05.mp4</a></p>

<p><strong>Straight episode 07</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep06.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep06.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep06.mp4</a></p>

<p><strong>Straight episode 08</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep07.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep07.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep07.mp4</a></p>

<p><strong>Straight episode 09</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep08.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep08.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep08.mp4</a></p>

<p><strong>Straight episode 10</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep09.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep09.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/straight/clean_unmodified_dense_straight_learned_policy_straight_ep09.mp4</a></p>

### Before Left Videos

Folder: `notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left`

<p><strong>Left episode 01</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep00.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep00.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep00.mp4</a></p>

<p><strong>Left episode 02</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep01.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep01.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep01.mp4</a></p>

<p><strong>Left episode 03</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep02.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep02.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep02.mp4</a></p>

<p><strong>Left episode 04</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep03.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep03.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep03.mp4</a></p>

<p><strong>Left episode 05</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep04.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep04.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep04.mp4</a></p>

<p><strong>Left episode 06</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep05.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep05.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep05.mp4</a></p>

<p><strong>Left episode 07</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep06.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep06.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep06.mp4</a></p>

<p><strong>Left episode 08</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep07.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep07.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep07.mp4</a></p>

<p><strong>Left episode 09</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep08.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep08.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep08.mp4</a></p>

<p><strong>Left episode 10</strong><br><video controls width="520" src="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep09.mp4"></video><br><a href="results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep09.mp4">notebooks/results/route_videos/clean_unmodified_dense_policy_10eps/left/clean_unmodified_dense_left_learned_policy_left_ep09.mp4</a></p>

## After Videos: Learned Creeping With Balanced Safety-Ellipse Shield

### After Straight Videos

Folder: `notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight`

<p><strong>Straight episode 01</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep00.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep00.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep00.mp4</a></p>

<p><strong>Straight episode 02</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep01.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep01.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep01.mp4</a></p>

<p><strong>Straight episode 03</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep02.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep02.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep02.mp4</a></p>

<p><strong>Straight episode 04</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep03.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep03.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep03.mp4</a></p>

<p><strong>Straight episode 05</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep04.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep04.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep04.mp4</a></p>

<p><strong>Straight episode 06</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep05.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep05.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep05.mp4</a></p>

<p><strong>Straight episode 07</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep06.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep06.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep06.mp4</a></p>

<p><strong>Straight episode 08</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep07.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep07.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep07.mp4</a></p>

<p><strong>Straight episode 09</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep08.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep08.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep08.mp4</a></p>

<p><strong>Straight episode 10</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep09.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep09.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/straight/final_balanced_safety_ellipse_straight_straight_ep09.mp4</a></p>

### After Left Videos

Folder: `notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left`

<p><strong>Left episode 01</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep00.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep00.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep00.mp4</a></p>

<p><strong>Left episode 02</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep01.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep01.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep01.mp4</a></p>

<p><strong>Left episode 03</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep02.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep02.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep02.mp4</a></p>

<p><strong>Left episode 04</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep03.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep03.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep03.mp4</a></p>

<p><strong>Left episode 05</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep04.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep04.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep04.mp4</a></p>

<p><strong>Left episode 06</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep05.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep05.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep05.mp4</a></p>

<p><strong>Left episode 07</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep06.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep06.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep06.mp4</a></p>

<p><strong>Left episode 08</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep07.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep07.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep07.mp4</a></p>

<p><strong>Left episode 09</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep08.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep08.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep08.mp4</a></p>

<p><strong>Left episode 10</strong><br><video controls width="520" src="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep09.mp4"></video><br><a href="results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep09.mp4">notebooks/results/route_videos/final_balanced_safety_ellipse_10eps/left/final_balanced_safety_ellipse_left_left_ep09.mp4</a></p>

## Next Fix Direction

The current shield is an ego-only intervention layer over a policy trained without that layer. To move toward 100%, the next technically clean step is not to tune the shield alone; it is to train or fine-tune inside the shielded dynamics so the policy learns to advance decisively when the shield permits entry. Separately, the reset distribution needs a documented safety-set check: if unsafe initial states remain in the benchmark, no ego-only controller can certify 100% success without changing the benchmark assumptions.

In [ ]:
# Verification cell: confirms the report is reading clean, unmodified-traffic artifacts.
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
REPO = ROOT.parent if ROOT.name == 'notebooks' else ROOT
RESULTS = REPO / 'notebooks/results/intersectionRouteAttentionCreeping'
for filename in ['clean_unmodified_dense_learned_vs_fast_summary.csv', 'final_before_after_balanced_safety_ellipse_summary.csv', 'uncontrollable_initial_collision_diagnostic.csv']:
    df = pd.read_csv(RESULTS / filename)
    assert (df['negotiating_traffic'].astype(str).str.lower() == 'false').all(), filename
print('clean artifact verification passed')